# Workbook 1 — In the misty beginnings: intelligence and alignment, then and now

**Exercises · Mysteries of Deep Learning, Iliad Intensive 2026**

Before deep learning, AI had three working pictures of intelligence: search over symbols, sequential decision and control, and statistical learning. This workbook looks at why the third took over, and at what alignment meant before and after capable systems came to be learned rather than written. Most sections are short guides to readings. The one derivation, planning as inference (§3), is a chain of exercises with worked answers; under stated assumptions, it gives planning and maximum-entropy RL a common formulation as inference in one graphical model.

**Route.** The sections are independent; start wherever interests you.
1. [Three pictures of intelligence](#wb1-section-1): a table of the three pictures and what each needs written by hand, and one question. Short.
2. [Why GOFAI failed](#wb1-section-2): one exponential blowup under two names, and Sutton's bitter lesson. Short, one core reading.
3. [Why statistical learning at all? Planning as inference](#wb1-section-3): five exercises, following Levine's tutorial, that derive the soft Bellman equation, show why exact inference makes a reckless agent, and recover maximum-entropy RL and KL-regularized control. It does not settle which route scales (see §3.C). The longest section; pen and paper help.
4. [Why alignment is hard, then](#wb1-section-4): the ontology problem as a short formal sketch, and natural abstractions. Short, one core reading.
5. [Why alignment is hard, now](#wb1-section-5): three readings on what a learned agent's values are, and a pointer to simulators in WB3. Readings only.

Readings marked *core* are the ones most worth your time within their section; everything else is optional. Worked answers to every question are in the solutions notebook, which is there for everyone to use.

<a id="wb1-section-1"></a>

## 1. Three pictures of intelligence

Classical AI had three formal pictures of intelligence, each needing one ingredient written by hand, and much of its history is a sequence of bets on which ingredient could instead be supplied by data and compute.

| picture | formal object | written by hand |
|---|---|---|
| search over symbols | state space $S$, operators, goal test, heuristic $h:S\to\mathbb R_{\ge0}$ (e.g. A* expands states in order of cost-so-far plus $h$) | $S$ and the operators, often $h$ |
| sequential decision and control | Markov decision process $(S,A,P,r,\gamma)$ and its Bellman optimality equation | the reward $r$; a Markov state space |
| statistical learning | hypothesis class $\mathcal H$, loss $\ell$, data distribution $\mathcal D$; empirical risk minimization $\hat h\in\arg\min_{h\in\mathcal H}\frac1n\sum_{i}\ell(h(x_i),y_i)$ (made precise in WB2) | $\mathcal H$: the architecture, and whatever shapes which functions are easy to reach |

**Question 1.1.** Place AlphaZero and a pretrained-then-RL-fine-tuned language model in this taxonomy. For each component (search, sequential decision, learned function), say what plays that role and what is still written by hand. Which hand-written ingredient did each system manage to replace with data?

<a id="wb1-section-2"></a>

## 2. Why GOFAI failed

A search tree with branching factor $b$ has $\sim b^d$ lines of play to depth $d$, and a tabular value function over $k$ state variables at $m$ levels each has $m^k$ entries. Lighthill's "combinatorial explosion" (1973) and Bellman's "curse of dimensionality" are the same blowup, and hand-written structure beats it only one problem at a time.

**[The bitter lesson](http://www.incompleteideas.net/IncIdeas/BitterLesson.html)**, Rich Sutton, 2019 · *core*

Across 70 years of AI research (chess, Go, speech, vision), general methods that leverage computation have beaten methods that build in human domain knowledge, by a large margin, because the cost of computation keeps falling exponentially. The two methods that "seem to scale arbitrarily" are search and learning. Two pages. The last two paragraphs carry the normative claim: build in the meta-methods that find structure, not the structure we have found.

**Question 2.1.** Sutton's vision paragraph credits modern networks that "use only the notions of convolution and certain kinds of invariances", while his final paragraph says we should stop building in simple ways to think about "space, objects, multiple agents, or symmetries". Current systems still contain a lot of hand-written structure: architectures (convolution, attention), tokenizers, curated data mixtures, RL reward rubrics, constitutions. Pick two. Is each a counterexample to the bitter lesson, or just the part that has not yet been eaten?

<a id="wb1-section-3"></a>

## 3. Why statistical learning at all? Planning as inference

**[Reinforcement learning and control as probabilistic inference: tutorial and review](https://arxiv.org/abs/1805.00909)**, Sergey Levine, 2018 · *core*

Maximum-entropy RL is equivalent to exact probabilistic inference in a graphical model with optimality variables when the dynamics are deterministic, and to variational inference when they are stochastic; maximum-entropy policy gradients, actor-critic methods and soft Q-learning then fall out as approximate inference. The exercises follow §2.2–2.4 and §3.1–3.2; §4 turns the result into soft Q-learning and actor-critic algorithms.

**The result.** Condition a model of trajectories on binary "optimality" variables with likelihood $e^{r}$. The posterior reweights the trajectory prior by $e^{R(\tau)}$; with deterministic dynamics and a fixed start state it is a Boltzmann distribution over feasible plans whose most probable trajectory is the classical optimal plan (Exercise 3.1). Its log-messages obey a *soft* Bellman equation (3.2), which under stochastic dynamics is optimistic (3.3). Holding the dynamics fixed gives variational inference, whose objective is maximum-entropy RL (3.4), and the one-step form is KL-regularized control, which has the form of a Bayesian update (3.5). This is a common formulation of planning and maximum-entropy RL under these assumptions. It does not make fitting a generative model to observations the same objective as control, which still needs a reward or goal.

The exercises build on each other, but each states what it uses. If you only do some, Exercises 3.3 and 3.5 have the most payoff.

### 3.A. The model

Take a finite-horizon MDP with finitely many states and actions (sums in place of Levine's integrals), and trajectories $\tau=(s_1,a_1,\dots,s_T,a_T)$. The "prior" over trajectories is the dynamics together with a uniform action prior $p(a\mid s)=1/|A|$:

$$
p(\tau)=p(s_1)\prod_{t=1}^{T-1}p(s_{t+1}\mid s_t,a_t)\,\prod_{t=1}^{T}\frac{1}{|A|}.
$$

At each step add a binary variable $\mathcal O_t$ ("step $t$ was optimal") with

$$
p(\mathcal O_t=1\mid s_t,a_t)=e^{\,r(s_t,a_t)} .
$$

For this to be a probability we need $r\le0$. Subtracting the maximum reward from every reward arranges that, and since every trajectory has $T$ steps it shifts every total reward by the same amount. (Levine §2.5 shows how to drop the assumption entirely.) Write $\mathcal O$ for the event $\mathcal O_1=\dots=\mathcal O_T=1$, write $\mathcal O_{t:T}$ for $\mathcal O_t=\dots=\mathcal O_T=1$, and let $R(\tau)=\sum_{t=1}^T r(s_t,a_t)$.

For comparison, the ordinary (hard) **Bellman optimality equation** for this finite-horizon, undiscounted problem is

$$
V^*_t(s)=\max_{a\in A}\Big[r(s,a)+\sum_{s'}p(s'\mid s,a)\,V^*_{t+1}(s')\Big],\qquad V^*_{T+1}\equiv0,
$$

the counterpart of the infinite-horizon discounted MDP equation $V^*(s)=\max_a\big[r(s,a)+\gamma\sum_{s'}P(s'\mid s,a)V^*(s')\big]$. Exercises 3.2(d) and 3.5(b) ask when the soft recursions reduce to it.

### 3.B. Five exercises

**Exercise 3.1 — the posterior over trajectories.**

(a) Show that $p(\tau\mid\mathcal O)\propto p(s_1)\prod_{t=1}^{T-1}p(s_{t+1}\mid s_t,a_t)\;e^{R(\tau)}$.

(b) Suppose the dynamics are deterministic, $s_{t+1}=f(s_t,a_t)$, and the start state is fixed. Describe $p(\tau\mid\mathcal O)$. Which trajectory is most probable?

(c) Introduce a temperature $\alpha>0$ by using $r/\alpha$ in place of $r$. What happens to the posterior in (b) as $\alpha\to0$?

<details><summary>Hint</summary>

Bayes' rule: $p(\tau\mid\mathcal O)=p(\mathcal O\mid\tau)\,p(\tau)/p(\mathcal O)$. Given the whole trajectory, each $\mathcal O_t$ depends only on $(s_t,a_t)$. Which factors are the same for every $\tau$?
</details>

**Exercise 3.2 — backward messages and the soft Bellman equation.** Define the backward messages

$$
\beta_t(s,a)=p(\mathcal O_{t:T}\mid s_t=s,a_t=a),\qquad \beta_t(s)=p(\mathcal O_{t:T}\mid s_t=s).
$$

(a) Show that $\beta_T(s,a)=e^{r(s,a)}$ and, for $t<T$,

$$
\beta_t(s,a)=e^{r(s,a)}\sum_{s'}p(s'\mid s,a)\,\beta_{t+1}(s'),\qquad \beta_t(s)=\frac{1}{|A|}\sum_a\beta_t(s,a).
$$

(b) Show that the posterior policy is $p(a_t=a\mid s_t=s,\mathcal O)=\beta_t(s,a)\big/\sum_{a'}\beta_t(s,a')$.

(c) Define $Q_T=r$ and, going backward,

$$
V_t(s)=\log\sum_a e^{Q_t(s,a)},\qquad Q_{t}(s,a)=r(s,a)+\log\sum_{s'}p(s'\mid s,a)\,e^{V_{t+1}(s')}.
$$

Show that $Q_t(s,a)=\log\beta_t(s,a)+c_t$ for a constant $c_t$ that does not depend on $s$ or $a$, and that the posterior policy is $e^{Q_t(s,a)-V_t(s)}$. These are Levine's soft $Q$- and value functions.

(d) Compare with the Bellman optimality equation in §3.A. Put in a temperature by using $r/\alpha$, and rescale $\tilde Q=\alpha Q$, $\tilde V=\alpha V$, so that

$$
\tilde V_t(s)=\alpha\log\sum_a e^{\tilde Q_t(s,a)/\alpha},\qquad \tilde Q_t(s,a)=r(s,a)+\alpha\log\sum_{s'}p(s'\mid s,a)\,e^{\tilde V_{t+1}(s')/\alpha}.
$$

What do the two soft operations become as $\alpha\to0$, first for deterministic dynamics and then for stochastic dynamics?

<details><summary>Hint for (a) and (b)</summary>

For (a), split $\mathcal O_{t:T}$ into $\mathcal O_t$ and $\mathcal O_{t+1:T}$ and sum over $s_{t+1}$. Given $(s_t,a_t)$, $\mathcal O_t$ is independent of everything later, and everything later depends on $(s_t,a_t)$ only through $s_{t+1}$. For (b), apply Bayes' rule to $a_t$ with $s_t$ fixed, then argue that once you know $s_t$, $\mathcal O_{1:t-1}$ is independent of $(a_t,\mathcal O_{t:T})$.
</details>

<details><summary>Hint for (c)</summary>

Induct backward from $t=T$. If $Q_{t+1}=\log\beta_{t+1}+c_{t+1}$, what is $V_{t+1}$ in terms of $\beta_{t+1}(s')$?
</details>

<details><summary>Hint for (d)</summary>

For numbers $x_1,\dots,x_n$ and weights $p_i>0$: $\;\max_i x_i+\alpha\log\min_i p_i\;\le\;\alpha\log\sum_i p_i e^{x_i/\alpha}\;\le\;\max_i x_i+\alpha\log\sum_i p_i$.
</details>

**Exercise 3.3 — inference is optimistic.** At some state the agent can take a *safe* action, which leads for certain to a state with soft value $0$, or a *gamble*, which leads with probability $p$ to a state with soft value $+R$ and otherwise to a state with soft value $-C$. Both actions have the same immediate reward, so ignore it. (Adding a constant to every value shifts both backups in (a) by that constant, so taking the safe value to be $0$ loses nothing.)

(a) Compute $Q(\text{gamble})$ under the exact-inference backup of Exercise 3.2, $\log\mathbb E\,e^{V}$, and under the ordinary expected backup, $\mathbb E\,V$.

(b) Evaluate both for $p=0.01$, $R=10$, $C=100$. Which action does each prefer, and by what odds does the exact posterior policy $e^{Q-V}$ pick the gamble over the safe action?

(c) Compute the posterior probability that the gamble came out well, $p(s_{t+1}=\text{good}\mid s_t,a_t=\text{gamble},\mathcal O)$. What has the agent "inferred"?

(d) For large $C$, when does exact inference prefer the gamble? How does the answer depend on $C$?

<details><summary>Hint for (c)</summary>

By Bayes' rule and the Markov property, $p(s_{t+1}\mid s_t,a_t,\mathcal O)\propto p(s_{t+1}\mid s_t,a_t)\,\beta_{t+1}(s_{t+1})$, and by Exercise 3.2(c), $\beta_{t+1}(s')\propto e^{V_{t+1}(s')}$.
</details>

**Exercise 3.4 — the variational fix is maximum-entropy RL.** Restrict to trajectory distributions that use the *true* start-state distribution and dynamics, and let only the policy vary. The horizon is finite, so the policy may depend on $t$:

$$
q(\tau)=p(s_1)\prod_{t=1}^{T-1}p(s_{t+1}\mid s_t,a_t)\,\prod_{t=1}^{T}\pi_t(a_t\mid s_t).
$$

(a) Show that

$$
\mathrm{KL}\big(q\,\|\,p(\cdot\mid\mathcal O)\big)=-\,\mathbb E_q\Big[\sum_{t=1}^T\Big(r(s_t,a_t)+\mathcal H\big(\pi_t(\cdot\mid s_t)\big)\Big)\Big]+T\log|A|+\log p(\mathcal O),
$$

where $\mathcal H(\pi_t(\cdot\mid s))=-\sum_a\pi_t(a\mid s)\log\pi_t(a\mid s)$ is the policy's entropy at $s$.

(b) Conclude that minimizing this KL over the policies is the same as maximizing expected reward plus policy entropy (maximum-entropy RL), and that the expectation minus $T\log|A|$ is a lower bound on $\log p(\mathcal O)$.

(c) When is the minimum KL zero, so that max-ent RL is *exact* inference? Use Exercise 3.3(c) to say why it is not, in general.

<details><summary>Hint</summary>

Write out $\log q(\tau)-\log p(\tau\mid\mathcal O)$ using the exact normalized posterior from Exercise 3.1(a),

$$
p(\tau\mid\mathcal O)=\frac{|A|^{-T}}{p(\mathcal O)}\,p(s_1)\prod_{t=1}^{T-1}p(s_{t+1}\mid s_t,a_t)\,e^{R(\tau)} .
$$

The start-state and dynamics factors cancel. For the policy term, $\mathbb E_q[\log\pi_t(a_t\mid s_t)]$ averages $\sum_a\pi_t(a\mid s_t)\log\pi_t(a\mid s_t)$ over the distribution of $s_t$.
</details>

**Exercise 3.5 — KL-regularized control as a Bayesian update.** Let $A$ be finite, let $\pi_0$ be a reference distribution on $A$ with full support, and let $r:A\to\mathbb R$.

(a) Let $Z=\sum_a\pi_0(a)e^{r(a)}$ and $\pi^*(a)=\pi_0(a)e^{r(a)}/Z$. Show that for every distribution $\pi$ on $A$,

$$
\mathbb E_\pi[r]-\mathrm{KL}(\pi\,\|\,\pi_0)=\log Z-\mathrm{KL}(\pi\,\|\,\pi^*).
$$

Conclude that $\max_\pi\{\mathbb E_\pi[r]-\mathrm{KL}(\pi\|\pi_0)\}=\log\mathbb E_{\pi_0}e^{r}$, attained only at $\pi^*\propto\pi_0e^{r}$. What changes for $\mathbb E_\pi[r]-\lambda\,\mathrm{KL}(\pi\|\pi_0)$ with a coefficient $\lambda>0$? (RLHF papers, and WB3, usually write this coefficient as $\beta$; here $\beta$ is taken by the backward messages.)

(b) Use (a) to solve the max-ent objective of Exercise 3.4 backward in time. Let $V_t(s)$ be the best achievable expected future reward-plus-entropy from state $s$ at time $t$, with $V_{T+1}\equiv0$. Show that the optimal policy is $\pi_t(a\mid s)=e^{Q_t(s,a)-V_t(s)}$, where

$$
Q_t(s,a)=r(s,a)+\mathbb E_{s'\sim p(\cdot\mid s,a)}V_{t+1}(s'),\qquad V_t(s)=\log\sum_a e^{Q_t(s,a)}.
$$

Compare with Exercise 3.2(c), redo Exercise 3.3(b) with this backup, and take the zero-temperature limit as in Exercise 3.2(d).

(c) Read $\pi^*\propto\pi_0e^{r}$ as Bayes' rule. What is the prior, what is the likelihood, and what is $Z$?

<details><summary>Hint for (a)</summary>

Expand both KLs as sums over $a$ and substitute $\log\pi^*(a)=\log\pi_0(a)+r(a)-\log Z$.
</details>

<details><summary>Hint for (b)</summary>

$\mathcal H(\pi)=\log|A|-\mathrm{KL}(\pi\,\|\,\mathrm{Unif}_A)$. Work from $t=T$ backward. Once the later policies are optimal, the part of the objective that depends on $\pi(\cdot\mid s_t)$ is $\mathbb E_{a\sim\pi}\big[r(s_t,a)+\mathbb E_{s'}V_{t+1}(s')\big]+\mathcal H(\pi)$.
</details>

### 3.C. Where this leaves statistical learning

In this model, planning does inference afresh at decision time from a known $p(s'\mid s,a)$, model-free RL amortizes the posterior into a policy trained from scalar rewards, and generative modelling fits a model of trajectories, including a prior over behaviour like the $\pi_0$ of Exercise 3.5, with no reward, leaving the conditioning on what we want (prompting, goal-conditioning, or the reweighting of Exercise 3.5) for later. The derivation does not say which of these makes better use of data and compute: WB3 §1 compares code lengths for labels and raw text, and WB2 §2 takes up the no-free-lunch objection to the learning route.

<a id="wb1-section-4"></a>

## 4. Why alignment is hard, then

Classical alignment meant writing a utility over the agent's world model; the ontology problem is that the model can change, leaving the old utility with several extensions.

**[Testing the natural abstraction hypothesis: project intro](https://www.lesswrong.com/posts/cy3BhHrGinZCp3LXE/testing-the-natural-abstraction-hypothesis-project-intro)**, John Wentworth, 2021 · *core*

Proposes the natural abstraction hypothesis: information about a system that matters far away has a low-dimensional summary (*abstractability*); humans use roughly those summaries (*human-compatibility*); and diverse learners converge on them (*convergence*). Argues alignment becomes much easier if it holds, and plans simulation tests. Short. WB5 asks related questions about representational convergence across networks.

**Setup for Question 4.1.** Let $U:S\to\mathbb R$ be a utility on the states of the agent's world model, and let a refined world model have states $S'$ with a map $\phi:S'\to S$ that is ambiguous on a region $E\subseteq S'$: each $s'\in E$ has two plausible images $\phi_1(s')\neq\phi_2(s')$. (For example, $U$ counts diamonds as lattices of carbon atoms, the refined model is about quantum fields, and $E$ holds edge cases such as exotic isotopes or simulated diamonds that the old ontology never had to classify.) Then $U_1=U\circ\phi_1$ and $U_2=U\circ\phi_2$ are both reasonable translations of $U$ to $S'$. They agree off $E$, may disagree on $E$, and nothing in $U$ chooses between them. An optimizer can still be drawn into $E$: if some states in $E$ are mapped by some $\phi_i$ to high-utility states but are cheaper to reach than ordinary ones (a simulated diamond is easier to make than a real one), then maximizing that $U_i$ over achievable states selects them, which is exactly where the choice of translation matters.

**Question 4.1.** In the diamond example, where does the ambiguity of $\phi$ come from? Which of Wentworth's three claims would have to hold for the ambiguity to be harmless, and is there residual ambiguity that even a true natural abstraction hypothesis would leave?

<a id="wb1-section-5"></a>

## 5. Why alignment is hard, now

Three readings on what a learned agent's values are; WB3 §6 adds a fourth, simulators and personae (janus; Nardo).

### 5.A. Values in a brain-like learner

**[Intro to brain-like AGI safety (sequence; posts 1, 3, 6 and 9)](https://www.lesswrong.com/s/HzcM2dkCq7fwXBej8)**, Steven Byrnes, 2022 (revised since) · *optional*

If AGI is built like the brain, as model-based RL with a learned world model plus a small hard-wired steering subsystem that supplies reward, then its values are a learned value function over concepts in its learned world model. They are neither its reward function nor reward-maximization, and alignment becomes a question of how reward and credit assignment shape them. For values, read [post 1](https://www.lesswrong.com/posts/4basF9w9jaPZpoC8R/intro-to-brain-like-agi-safety-1-what-s-the-problem-and-why) (the problem), [post 3](https://www.lesswrong.com/posts/hE56gYi5d68uux9oM/intro-to-brain-like-agi-safety-3-two-subsystems-learning-and) (learning and steering subsystems), [post 6](https://www.lesswrong.com/posts/qNZSBqLEh4qLRqgWW/intro-to-brain-like-agi-safety-6-big-picture-of-motivation) (motivation and decision-making) and [post 9](https://www.lesswrong.com/posts/vpdJz4k5BgGzuGo7A/intro-to-brain-like-agi-safety-9-takeaways-from-neuro-2-2-on) (AGI motivation; its §9.2, goals defined over latent variables, is §4's ontology problem seen from inside a learner).

### 5.B. Learned optimization

**[Risks from learned optimization in advanced machine learning systems](https://arxiv.org/abs/1906.01820)**, Evan Hubinger, Chris van Merwijk, Vladimir Mikulik, Joar Skalse, Scott Garrabrant, 2019 · *core*

A trained model may itself be an optimizer (a mesa-optimizer) whose objective differs from the training loss. This splits alignment into outer alignment (loss vs intent) and inner alignment (mesa-objective vs loss); a pseudo-aligned model agrees with the loss on the training distribution and comes apart off it. §1 defines the terms; §2 asks which tasks and base optimizers favour mesa-optimization (§2.1 argues that diverse environments favour learned search: §2's bitter lesson applied *inside* the model); §4 on deceptive alignment is optional; §7 is a glossary.

### 5.C. "Just retarget the search"

**[How to go from interpretability to alignment: just retarget the search](https://www.lesswrong.com/posts/w4aeAFzSAguvqA5qu/how-to-go-from-interpretability-to-alignment-just-retarget-the-search)**, John Wentworth, 2022 · *optional*

Assume (i) some version of the natural abstraction hypothesis, so the model has an internal concept for an alignment target such as human values or user intent, and (ii) the mesa-optimization argument, so the model contains retargetable general-purpose search. Then interpretability could find the search's target and point it at that concept, which removes the inner alignment problem but leaves outer alignment unsolved. Short; read all of it, including the 'Problems' section.

### What carries forward

- **The statistical-learning picture** of §1 is made precise in WB2; WB2 §2 takes up the no-free-lunch objection mentioned in §3.C.
- **The reweighting $\pi^*\propto\pi_0e^{r/\lambda}$** of Exercise 3.5 (with $\lambda$ written $\beta$) is applied to RLHF in WB3 §6, which also develops the simulator picture pointed to in §5.
- **Natural abstractions** (§4) connect to the questions about representational convergence in WB5.
- **The hand-written part of $\mathcal H$** in §1, its inductive bias, is taken apart in WB4.

### Further study

- Levine §5 reviews earlier work connecting inference and control (the Kalman duality, linearly solvable MDPs, KL-constrained policy search) and applications such as intent inference.
- The rest of Byrnes's sequence (posts 10–15) covers the technical alignment problem and two paths forward ("controlled AGI" and "social-instinct AGI").